# EX_01 — Embeddings básicos (ejercicios)

**Notebook de referencia:** `notebook/01_Introduccion_NLP_Embeddings_Basicos.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — One-hot desde vocabulario

Dado un vocabulario fijo y una frase tokenizada en palabras, construye una matriz one-hot `(seq_len, vocab_size)` sin sklearn (solo NumPy).


In [1]:
import numpy as np

vocab = ["cat", "sat", "mat", "the"]
# TODO: try another sentence (puedes dejar la original o cambiarla)
sentence = ["the", "cat", "sat"] 

# =====================================================================
# 1. TODO: build index (Mapear cada palabra a su número de índice)
# =====================================================================
# Creamos un diccionario para saber rápidamente qué posición tiene cada palabra
# {"cat": 0, "sat": 1, "mat": 2, "the": 3}
word_to_index = {word: i for i, word in enumerate(vocab)}


# =====================================================================
# 2. TODO: build one-hot matrix
# =====================================================================
seq_len = len(sentence)
vocab_size = len(vocab)

# Creamos una matriz de ceros con la forma deseada (seq_len, vocab_size)
one_hot_matrix = np.zeros((seq_len, vocab_size), dtype=np.float32)

# Rellenamos con un 1 la posición correcta de cada palabra
for row_idx, word in enumerate(sentence):
    if word in word_to_index:
        col_idx = word_to_index[word]
        one_hot_matrix[row_idx, col_idx] = 1.0

# --- Verificación de resultados ---
print("Estructura del vocabulario (Índices):", word_to_index)
print(f"\nFrase a transformar: {sentence}\n")
print("Matriz One-hot resultante con dimensiones (seq_len, vocab_size):")
print(one_hot_matrix)


Estructura del vocabulario (Índices): {'cat': 0, 'sat': 1, 'mat': 2, 'the': 3}

Frase a transformar: ['the', 'cat', 'sat']

Matriz One-hot resultante con dimensiones (seq_len, vocab_size):
[[0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]]


## Actividad 2 — Similitud coseno

Implementa `cosine_similarity(a, b)` para vectores 1D y compara dos palabras usando sus filas one-hot (esperado: 0 o 1). Luego discute por qué one-hot no captura similitud semántica.


In [2]:
import numpy as np

# =====================================================================
# 1. TODO: cosine_similarity (Implementación matemática usando NumPy)
# =====================================================================
def cosine_similarity(a, b):
    # np.dot calcula el producto escalar (a . b)
    dot_product = np.dot(a, b)
    
    # np.linalg.norm calcula la norma/magnitud euclidiana del vector
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    
    # Evitamos división por cero por si acaso un vector es todo ceros
    if norm_a == 0 or norm_b == 0:
        return 0.0
        
    return dot_product / (norm_a * norm_b)


# =====================================================================
# 2. TODO: pick two rows from your one-hot matrix
# =====================================================================
# Reutilizamos vectores One-hot idénticos a los del ejercicio anterior:
# "cat" -> [1, 0, 0, 0]
# "sat" -> [0, 1, 0, 0]
word_cat = np.array([1.0, 0.0, 0.0, 0.0])
word_sat = np.array([0.0, 1.0, 0.0, 0.0])

# Calculamos similitud entre dos palabras distintas ("cat" vs "sat")
sim_distintas = cosine_similarity(word_cat, word_sat)

# Calculamos similitud de una palabra consigo misma ("cat" vs "cat")
sim_identicas = cosine_similarity(word_cat, word_cat)


# --- Impresión de resultados ---
print(f"Similitud coseno entre 'cat' y 'sat' (Diferentes): {sim_distintas}")
print(f"Similitud coseno entre 'cat' y 'cat' (Idénticas): {sim_identicas}")

Similitud coseno entre 'cat' y 'sat' (Diferentes): 0.0
Similitud coseno entre 'cat' y 'cat' (Idénticas): 1.0


## Actividad 3 — Ventana con Gensim (lectura + entrenamiento mínimo)

Entrena un `Word2Vec` minúsculo sobre `sentences` (lista de listas de tokens) con `vector_size=8`, `window=2`, `min_count=1`. Imprime el vector de una palabra y la similitud entre dos.

*Hint:* `from gensim.models import Word2Vec`.


In [3]:
import torch
import torch.nn as nn
import numpy as np

# --- CONFIGURACIÓN INICIAL ---
vocab_size = 4  # ["cat", "sat", "mat", "the"]
embedding_dim = 5  # Cada palabra será representada por 5 números

# 1. Creamos una matriz de pesos aleatorios (nuestro espacio de embeddings simulado)
# Tamaño: (vocab_size, embedding_dim) -> (4, 5)
E_weight = torch.randn(vocab_size, embedding_dim)

# Definimos el índice de la palabra que queremos buscar (por ejemplo, el índice 3 que corresponde a "the")
word_idx = 3


# =====================================================================
# 1. TODO: One-hot dot product (Búsqueda matemática conceptual)
# =====================================================================
# Para extraer la fila usando multiplicación, creamos el vector One-hot de la palabra
one_hot = torch.zeros(vocab_size)
one_hot[word_idx] = 1.0  # tensor([0., 0., 0., 1.])

# La teoría dice que: Vector_OneHot (1, 4) x Matriz_Embeddings (4, 5) = Vector_Embedding (1, 5)
# Usamos torch.matmul para multiplicar matrices (o el operador @)
embedding_via_matmul = one_hot @ E_weight


# =====================================================================
# 2. TODO: nn.Embedding (Uso eficiente en PyTorch)
# =====================================================================
# Instanciamos la capa oficial de PyTorch
emb_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

# Para que los resultados coincidan exactamente y puedas verificarlo, 
# le asignamos manualmente a la capa los mismos pesos 'E_weight' que creamos arriba
emb_layer.weight.data = E_weight

# Pasamos el índice como un tensor de PyTorch
idx_tensor = torch.tensor([word_idx])
embedding_via_layer = emb_layer(idx_tensor)


# --- COMPROBACIÓN Y VERIFICACIÓN ---
print("Matriz original de Embeddings (E_weight):\n", E_weight)
print(f"\nExtrayendo la fila para el índice {word_idx}:")
print("\n[Método 1] Mediante multiplicación One-hot:\n", embedding_via_matmul)
print("\n[Método 2] Mediante capa nn.Embedding de PyTorch:\n", embedding_via_layer.squeeze(0))

# Verificamos que ambos métodos den exactamente el mismo resultado
np.testing.assert_allclose(
    embedding_via_matmul.detach().numpy(), 
    embedding_via_layer.squeeze(0).detach().numpy(), 
    rtol=1e-5
)
print("\n¡Excelente! Ambos métodos extraen exactamente el mismo vector de características.")


Matriz original de Embeddings (E_weight):
 tensor([[ 0.2837, -0.2740, -1.0570,  1.8864, -0.1932],
        [ 0.1772, -1.1369,  1.0118,  2.3961,  0.5907],
        [-0.8056, -0.3055,  0.1444, -1.2508,  0.1373],
        [-0.3570, -0.6366,  0.4186,  0.8368,  1.3734]])

Extrayendo la fila para el índice 3:

[Método 1] Mediante multiplicación One-hot:
 tensor([-0.3570, -0.6366,  0.4186,  0.8368,  1.3734])

[Método 2] Mediante capa nn.Embedding de PyTorch:
 tensor([-0.3570, -0.6366,  0.4186,  0.8368,  1.3734],
       grad_fn=<SqueezeBackward1>)

¡Excelente! Ambos métodos extraen exactamente el mismo vector de características.
